# Map or Ritual? Release notebook, part B (CPU)

Recomputes every table, contrast and figure in the paper from per-episode outcomes, and ships the probe-agent source.

The embedded table has one row per episode (model, task set, task id, condition, seed, hit@5, hit@1, file@5, tool counts, status). A second table adds derived scores for the primary study (reachability flag, whether a graph tool surfaced a gold function, whether the agent followed it up, and hit@5 under the audit label sets), with per-task gold-node counts. It holds no competition source code, issue text, labels or tool output. Raw traces are not published because tool outputs quote competition source files.

Part A (labels and worlds, hash-checked) and the held-out builders are separate notebooks. No GPU, no internet, no input needed. License: Apache 2.0.

In [ ]:
# ===== DATA: per-episode outcomes, embedded =====
# One row per agent episode: model, task set, task id, condition, seed, hit@5, hit@1,
# file@5, tool calls, graph calls, status. No competition source code, issue text or
# tool output is included. Raw traces are not published because tool outputs quote
# competition source files.
import base64, gzip, hashlib, json
_B64 = "..."
_blob = gzip.decompress(base64.b64decode(_B64))
OUTCOMES_SHA256 = hashlib.sha256(_blob).hexdigest()
assert OUTCOMES_SHA256[:16] == "00a9ba201be06f31", "embedded data changed"
D = json.loads(_blob)
print("episodes:", len(D["episodes"]), "| review rows:", len(D["review"]), "| official rows:", len(D["official"]))
print("outcomes sha256:", OUTCOMES_SHA256[:16])

In [ ]:
# ===== ANALYSIS: every number in the paper's results =====
import collections
import numpy as np
import pandas as pd
E = pd.DataFrame(D["episodes"], columns=D["episodes_columns"])

def task_means(df, model, sets, cond):
    x = df[(df.model == model) & (df.task_set.isin(sets)) & (df.condition == cond)]
    return x.groupby("task_id").hit5.mean()

def contrast(model, sets, a, b, seed=0, B=10000):
    ta, tb = task_means(E, model, sets, a), task_means(E, model, sets, b)
    ids = sorted(set(ta.index) & set(tb.index))
    d = np.array([ta[i] - tb[i] for i in ids])
    bs = d[np.random.default_rng(seed).integers(0, len(d), (B, len(d)))].mean(1)
    lo, hi = np.percentile(bs, [2.5, 97.5]); l9, h9 = np.percentile(bs, [5, 95])
    return dict(n=len(ids), diff=100 * d.mean(), lo=100 * lo, hi=100 * hi, equivalent=bool(l9 > -0.05 and h9 < 0.05))

def fmt(r):
    return f"{r['diff']:+5.1f} [{r['lo']:+5.1f}, {r['hi']:+5.1f}] n={r['n']}" + ("  equivalent" if r["equivalent"] else "")

PRIMARY = ["primary"]                       # pre-specified study, seeds 1 and 2
PRIMARY8 = ["primary", "primary_extra_seeds"]
COMP = ["primary", "primary_extra_seeds", "secondary"]
HELD = ["heldout_issues"]

print("=== Table 1. hit@5 (%) on the 82 primary tasks, seeds 1-2 ===")
for m in ("E4B", "31B"):
    print(m, "  ".join(f"{c}={100 * task_means(E, m, PRIMARY, c).mean():.1f}"
                       for c in ["NR", "W0", "W1", "W2", "W3", "W1G", "W2G", "W3G"]))

print("\n=== Pre-specified contrasts (82 primary tasks, seeds 1-2) ===")
for m in ("E4B", "31B"):
    for name, a, b in (("H2 W1G-W2G", "W1G", "W2G"), ("H4 W1G-W3G", "W1G", "W3G"), ("H5 W1-W2", "W1", "W2"),
                       ("grep vs guess W0-NR", "W0", "NR"), ("graph vs grep W1-W0", "W1", "W0")):
        print(f"{m} {name:20s}", fmt(contrast(m, PRIMARY, a, b)))

print("\n=== Table 2. Competition (120 tasks, all seeds) and held-out (issue tasks) ===")
for name, a, b in (("real tools - grep", "W1", "W0"), ("rewired tools - grep", "W2", "W0"),
                   ("real - rewired, neutral", "W1", "W2"), ("real - rewired, graph-first", "W1G", "W2G")):
    for m in ("E4B", "31B"):
        comp = fmt(contrast(m, COMP, a, b))
        held = fmt(contrast(m, HELD, a, b)) if (E.task_set == "heldout_issues").any() else "not available"
        print(f"{name:28s} {m}  competition {comp}  |  held-out {held}")

print("\n=== E4B extension seeds (82 primary tasks, seeds 1-8) ===")
for a, b in (("W1G", "W2G"), ("W1", "W2")):
    print(f"{a}-{b}", fmt(contrast("E4B", PRIMARY8, a, b)))

print("\n=== Adoption: share of episodes with at least one graph call ===")
for m in ("E4B", "31B"):
    for sets, lab in ((PRIMARY, "primary"), (HELD, "held-out")):
        x = E[(E.model == m) & E.task_set.isin(sets)]
        if len(x):
            print(m, lab, "  ".join(f"{c}={100 * (x[x.condition == c].graph_calls > 0).mean():.1f}%" for c in ["W1", "W2", "W1G", "W2G"]))

R = pd.DataFrame(D["review"], columns=D["review_columns"])
print("\n=== Review turn (exploratory) ===")
for m in ("E4B", "31B"):
    x = R[R.model == m]
    L = x[x.arm == "L"].groupby("task_id").hit5.mean(); P = x[x.arm == "P"].groupby("task_id").hit5.mean()
    d = (L - P).values
    bs = d[np.random.default_rng(0).integers(0, len(d), (10000, len(d)))].mean(1)
    print(m, f"list minus plain reminder {100 * d.mean():+.1f} [{100 * np.percentile(bs, 2.5):+.1f}, {100 * np.percentile(bs, 97.5):+.1f}]")

O = pd.DataFrame(D["official"], columns=D["official_columns"])
print("\n=== Official sample agent, E4B, bug fixing ===")
for w in ("W1", "W2"):
    x = O[O.condition == w]
    print(w, "runs", len(x), "resolved", int(x.resolved.sum()), "with patch", int(x.has_patch.sum()),
          "used an edge tool", int((x.edge_calls > 0).sum()))
print("ANALYSIS DONE")

In [ ]:
# ===== MORE NUMBERS: H3, secondary, mechanism, label audit, GEE, held-out details =====
X = pd.DataFrame(D["primary_extra"], columns=D["primary_extra_columns"])
def boot_ci(d, seed=0, B=10000):
    d = np.asarray(d, float); bs = d[np.random.default_rng(seed).integers(0, len(d), (B, len(d)))].mean(1)
    lo, hi = np.percentile(bs, [2.5, 97.5]); l9, h9 = np.percentile(bs, [5, 95])
    return f"{100 * d.mean():+.1f} [{100 * lo:+.1f}, {100 * hi:+.1f}]" + ("  equivalent" if l9 > -0.05 and h9 < 0.05 else "")

print("=== H3: W1G-W2G gap, reachable minus unreachable tasks (82 primary) ===")
for m in ("E4B", "31B"):
    a, b = task_means(E, m, PRIMARY, "W1G"), task_means(E, m, PRIMARY, "W2G")
    reach = X[X.model == m].groupby("task_id").reach.first()
    ids = sorted(a.index); arr = np.array([a[i] - b[i] for i in ids]); rc = np.array([bool(reach[i]) for i in ids])
    rng = np.random.default_rng(0); bs = []
    for _ in range(10000):
        j = rng.integers(0, len(ids), len(ids)); r_ = rc[j]
        if r_.all() or (~r_).all(): continue
        bs.append(arr[j][r_].mean() - arr[j][~r_].mean())
    print(m, f"reachable {rc.sum()} | H3 {100 * (arr[rc].mean() - arr[~rc].mean()):+.1f} [{100 * np.percentile(bs, 2.5):+.1f}, {100 * np.percentile(bs, 97.5):+.1f}]")

print("\n=== Secondary set (38 tasks): E4B real tools minus grep ===")
print("E4B W1-W0", fmt(contrast("E4B", ["secondary"], "W1", "W0")))
sec_ids = set(E[E.task_set == "secondary"].task_id)
print("secondary composition", dict(collections.Counter(D["task_kind"][i] for i in sec_ids)))

print("\n=== Mechanism: graph tool returned a gold function (82 primary) ===")
for m in ("E4B", "31B"):
    x = X[(X.model == m) & (X.surfaced >= 0)]
    s = {c: x[x.condition == c].groupby("task_id").surfaced.mean() for c in ("W1G", "W2G")}
    print(m, f"W1G {100 * x[x.condition == 'W1G'].surfaced.mean():.0f}%  W2G {100 * x[x.condition == 'W2G'].surfaced.mean():.0f}%  diff",
          boot_ci([s["W1G"][i] - s["W2G"][i] for i in sorted(s["W1G"].index)]))
    f = x[x.followed_up >= 0]
    print("   followed up after surfacing:", "  ".join(f"{c}={100 * f[f.condition == c].followed_up.mean():.0f}%" for c in ("W1", "W2", "W3", "W1G", "W2G", "W3G")))

print("\n=== Label audit: primary gold nodes and re-scoring ===")
cnt = np.array(list(D["audit_counts"].values()))
print(f"text-matching nodes {cnt[:, 0].sum()} | AST nodes {cnt[:, 1].sum()} | both {cnt[:, 2].sum()} ({100 * cnt[:, 2].sum() / cnt[:, 0].sum():.1f}%)")
for lab in ("hit5_ast", "hit5_both", "hit5_union"):
    for m in ("E4B", "31B"):
        x = X[(X.model == m) & (X[lab] >= 0)]
        t = {c: x[x.condition == c].groupby("task_id")[lab].mean() for c in ("W1G", "W2G", "W1", "W2", "W3G")}
        ids = sorted(t["W1G"].index)
        print(f"{lab:10s} {m}  H2 {boot_ci([t['W1G'][i] - t['W2G'][i] for i in ids])}  |  H5 {boot_ci([t['W1'][i] - t['W2'][i] for i in ids])}  |  H4 {boot_ci([t['W1G'][i] - t['W3G'][i] for i in ids])}")

print("\n=== GEE (task clusters, exchangeable): odds ratio real vs rewired, W1/W2/W1G/W2G, 82 primary ===")
import statsmodels.api as sm, statsmodels.formula.api as smf, warnings
warnings.filterwarnings("ignore")
for m in ("E4B", "31B"):
    d = E[(E.model == m) & (E.task_set == "primary") & E.condition.isin(["W1", "W2", "W1G", "W2G"])].copy()
    d["real"] = d.condition.str.startswith("W1").astype(int); d["gf"] = d.condition.str.endswith("G").astype(int)
    g = smf.gee("hit5 ~ real*gf", groups="task_id", data=d, family=sm.families.Binomial(), cov_struct=sm.cov_struct.Exchangeable()).fit()
    print(m, "OR real", np.exp(g.params["real"]).round(2), np.exp(g.conf_int().loc["real"]).round(2).tolist())

print("\n=== Held-out issues: graph-first over grep, and both prompts averaged ===")
for m in ("E4B", "31B"):
    print(m, "W1G-W0", fmt(contrast(m, HELD, "W1G", "W0")), "| W2G-W0", fmt(contrast(m, HELD, "W2G", "W0")))
for m in ("E4B", "31B"):
    for sets, lab in ((COMP, "competition"), (HELD, "held-out")):
        t = {c: task_means(E, m, sets, c) for c in ("W1", "W1G", "W2", "W2G")}
        ids = sorted(set(t["W1"].index) & set(t["W2"].index) & set(t["W1G"].index) & set(t["W2G"].index))
        print(m, lab, "real minus rewired, prompts averaged", boot_ci([(t["W1"][i] + t["W1G"][i] - t["W2"][i] - t["W2G"][i]) / 2 for i in ids]))

print("\n=== First held-out version (commit messages, 120 tasks) ===")
for m in ("E4B", "31B"):
    print(m, "W1-W0", fmt(contrast(m, ["heldout_commits"], "W1", "W0")), "| W2-W0", fmt(contrast(m, ["heldout_commits"], "W2", "W0")))

print("\n=== Graph-first episodes with exactly one graph call (82 primary) ===")
for m in ("E4B", "31B"):
    x = E[(E.model == m) & (E.task_set == "primary") & E.condition.isin(["W1G", "W2G", "W3G"])]
    print(m, f"{100 * (x.graph_calls == 1).mean():.1f}%")

print("\n=== Official agent: resolved real minus rewired, and tool calls ===")
O = pd.DataFrame(D["official"], columns=D["official_columns"])
r1 = O[O.condition == "W1"].set_index("task_id").resolved; r2 = O[O.condition == "W2"].set_index("task_id").resolved
print("resolved W1-W2", boot_ci([r1[i] - r2[i] for i in sorted(r1.index)]),
      "| similarity calls", int(O.similarity_calls.sum()), "| edge calls", int(O.edge_calls.sum()),
      "| runs with an edge call", f"{100 * (O.edge_calls > 0).mean():.0f}%")
print("MORE NUMBERS DONE")

In [ ]:
# ===== FIGURES 1 and 2 (600 DPI PNG and vector PDF in /kaggle/working) =====
import os
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."

def tm_from_table(model, sets):
    x = E[(E.model == model) & E.task_set.isin(sets)]
    return {c: g.groupby("task_id").hit5.mean().to_dict() for c, g in x.groupby("condition")}

def diff(t, a, b, seed=0):
    ids = sorted(set(t[a]) & set(t[b]))
    d = np.array([t[a][i] - t[b][i] for i in ids])
    bs = d[np.random.default_rng(seed).integers(0, len(d), (10000, len(d)))].mean(1)
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return 100 * d.mean(), 100 * lo, 100 * hi

sets = {("E4B", "Competition"): tm_from_table("E4B", COMP), ("31B", "Competition"): tm_from_table("31B", COMP)}
for m in ("E4B", "31B"):
    if ((E.model == m) & (E.task_set == "heldout_issues")).any():
        sets[(m, "Held-out")] = tm_from_table(m, HELD)
SURFACE, INK, INK2, GRID = "#ffffff", "#000000", "#333333", "#e6e6e6"
REAL, REWIRED = "#1f3b73", "#a6a6a6"   # navy and gray; rewired also carries a hatch, so it reads in grayscale
DPI = 600
plt.rcParams.update({"font.family": "serif", "font.serif": ["STIXGeneral", "DejaVu Serif"],
                     "mathtext.fontset": "stix", "font.size": 9, "axes.linewidth": 0.6,
                     "axes.edgecolor": INK, "axes.labelcolor": INK, "xtick.color": INK, "ytick.color": INK,
                     "xtick.direction": "in", "ytick.direction": "in", "xtick.major.width": 0.6,
                     "ytick.major.width": 0.6, "xtick.major.size": 3, "ytick.major.size": 3,
                     "legend.fontsize": 8, "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
                     "savefig.facecolor": SURFACE, "pdf.fonttype": 42, "hatch.linewidth": 0.5})

def load(f):
    last = {}
    for l in open(f):
        l = l.strip()
        if l:
            r = json.loads(l)
            last[(r["id"], r["world"], r["seed"])] = r
    return last

def taskmeans(D):
    t = collections.defaultdict(lambda: collections.defaultdict(list))
    for (i, w, s), r in D.items():
        t[w][i].append(float(bool(r["hit5"])))
    return {w: {i: np.mean(v) for i, v in d.items()} for w, d in t.items()}

def diff(t, a, b, seed=0):
    ids = sorted(set(t[a]) & set(t[b]))
    d = np.array([t[a][i] - t[b][i] for i in ids])
    rng = np.random.default_rng(seed)
    bs = d[rng.integers(0, len(d), (10000, len(d)))].mean(1)
    lo, hi = np.percentile(bs, [2.5, 97.5])
    return 100 * d.mean(), 100 * lo, 100 * hi



order = [k for k in [("E4B", "Competition"), ("E4B", "Held-out"), ("31B", "Competition"), ("31B", "Held-out")] if k in sets]
labels = [f"{m}\n{d.lower()}" for m, d in order]
rows = {}

# ---------- Figure 1: gain over grep, real vs rewired, neutral prompt ----------
fig, ax = plt.subplots(figsize=(6.8, 3.3))
x = np.arange(len(order))
w = 0.30
for j, (a, col, name) in enumerate((("W1", REAL, "Real graph tools"), ("W2", REWIRED, "Rewired graph tools"))):
    vals = [diff(sets[k], a, "W0") for k in order]
    rows[name] = vals
    xs = x + (j - 0.5) * (w + 0.06)
    ax.bar(xs, [v[0] for v in vals], width=w, color=col, zorder=3, edgecolor=INK, linewidth=0.5,
           hatch=None if a == "W1" else "////")
    ax.errorbar(xs, [v[0] for v in vals], yerr=[[v[0] - v[1] for v in vals], [v[2] - v[0] for v in vals]],
                fmt="none", ecolor=INK, elinewidth=0.7, capsize=2, capthick=0.7, zorder=4)
    for xi, v in zip(xs, vals):
        ax.text(xi, v[2] + 1.1, f"{v[0]:+.1f}".replace("-", "\u2212"), ha="center", va="bottom", fontsize=7.5,
                color=INK, zorder=6, bbox=dict(facecolor=SURFACE, edgecolor="none", pad=0.6))
ax.axhline(0, color=INK, linewidth=0.6, zorder=2)
ax.set_xticks(x, labels)
top = max(r[2] for k in ("W1", "W2") for r in [diff(sets[o], k, "W0") for o in order])
ax.set_ylim(-7.5, top + 4.5)
ax.set_ylabel("Gain in hit@5 over grep only (pp)", labelpad=10)
ax.yaxis.set_label_coords(-0.075, 0.5)
for s in ("top", "right"):
    ax.spines[s].set_visible(False)
ax.legend(handles=[Patch(facecolor=REAL, edgecolor=INK, linewidth=0.5, label="Real graph (W1 $-$ W0)"),
                   Patch(facecolor=REWIRED, edgecolor=INK, linewidth=0.5, hatch="////", label="Rewired graph (W2 $-$ W0)")],
          frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.02), ncol=2, columnspacing=2.5,
          handlelength=2.2)
fig.tight_layout()
fig.savefig(f"{OUT}/fig1_gain_over_grep.png", dpi=DPI)
fig.savefig(f"{OUT}/fig1_gain_over_grep.pdf")
plt.close(fig)

# ---------- Figure 2: real minus rewired, both prompts, equivalence band ----------
fig, ax = plt.subplots(figsize=(6.8, 3.4))
ax.axvspan(-5, 5, color="#efefef", zorder=0, linewidth=0)
ax.axvline(0, color=INK, linewidth=0.6, zorder=1)
y = np.arange(len(order))[::-1]
for j, (a, b, col, mk, name) in enumerate((("W1", "W2", INK, "o", "Neutral prompt"),
                                            ("W1G", "W2G", "#555555", "s", "Graph-first prompt"))):
    vals = [diff(sets[k], a, b) for k in order]
    rows[name] = vals
    ys = y + (0.17 if j == 0 else -0.17)
    ax.hlines(ys, [v[1] for v in vals], [v[2] for v in vals], color=col, linewidth=0.9, zorder=3)
    ax.plot([v[0] for v in vals], ys, mk, markersize=4.5, markeredgecolor=col, markeredgewidth=0.9,
            markerfacecolor=col if j == 0 else SURFACE, zorder=4)
ax.set_yticks(y, [l.replace("\n", ", ") for l in labels])
ax.set_xlabel("Real minus rewired edges, difference in hit@5 (pp)", labelpad=8)
ax.set_xlim(-14, 14)
ax.set_ylim(-0.5, len(order) + 0.45)
ax.text(-4.7, len(order) + 0.3, "equivalence\nmargin $\\pm$5 pp", ha="left", va="top", fontsize=7.5,
        color=INK2, linespacing=1.15)
for sp in ("top", "right", "left"):
    ax.spines[sp].set_visible(False)
ax.tick_params(axis="y", length=0, pad=10)
ax.tick_params(axis="x", pad=5)
ax.legend(handles=[Line2D([], [], color=INK, marker="o", markersize=4.5, linewidth=0.9, label="Neutral prompt (W1 $-$ W2)"),
                   Line2D([], [], color="#555555", marker="s", markersize=4.5, markerfacecolor=SURFACE, linewidth=0.9,
                          label="Graph-first prompt (W1G $-$ W2G)")],
          frameon=False, loc="lower center", bbox_to_anchor=(0.5, 1.06), ncol=2, columnspacing=2.5)
fig.tight_layout()
fig.savefig(f"{OUT}/fig2_real_minus_rewired.png", dpi=DPI)
fig.savefig(f"{OUT}/fig2_real_minus_rewired.pdf")
plt.close(fig)

print("FIGURES DONE:", sorted(f for f in os.listdir(OUT) if f.startswith("fig")))

In [ ]:
# ===== PROBE HARNESS SOURCE (reference, needs 4 GPUs and the official vLLM setup) =====
# Written to probe_harness.py for reading. Not executed here.
PROBE_SOURCE = "# ===== PROBE PART 1: data, worlds, read-only sandbox =====\nimport os, re, json, glob, time, random, threading, subprocess\nimport collections\nfrom pathlib import Path\n\nDATA_DIR = Path('/kaggle/input/competitions/gemma-4-developer-agent')\nA = os.path.dirname(glob.glob('/kaggle/input/*/*/*/labels_v1.json')[0])\nSRCROOT = f\"{A}/src\"\nW = f\"{A}/worlds\"\nEMB = str(DATA_DIR / 'embeddings')\nWORLDS = {\n    \"W1\": (f\"{W}/W1/graphs\", EMB),\n    \"W2\": (f\"{W}/W2/graphs\", EMB),\n    \"W3\": (f\"{W}/W1/graphs\", f\"{W}/W3/embeddings\"),\n}\nTASKS = {}\nwith open(DATA_DIR / 'tasks.jsonl') as f:\n    for line in f:\n        d = json.loads(line)\n        TASKS[d[\"instance_id\"]] = d\nwith open(f\"{A}/labels_v1.json\") as f:\n    LABELS = {r[\"id\"]: r for r in json.load(f)}\n\ndef clip(s, n=3000):\n    s = str(s)\n    if len(s) <= n:\n        return s\n    return s[:n] + f\"\\n...[truncated {len(s) - n} chars]\"\n\ndef strip(n):\n    return str(n).split(\";\", 1)[-1].strip()\n\nALLOWED = {\"grep\", \"egrep\", \"fgrep\", \"rg\", \"find\", \"ls\", \"cat\",\n           \"head\", \"tail\", \"sed\", \"wc\", \"sort\", \"uniq\", \"cut\"}\nBAD = (\">\", \"`\", \"$(\", \";\", \"\\n\")\n\ndef run_command(iid, command):\n    cmd = str(command).strip()\n    if any(b in cmd for b in BAD):\n        return \"ERROR: only read-only commands allowed (no ; > ` $( ).\"\n    for seg in re.split(r\"\\|\\||&&|\\|\", cmd):\n        toks = seg.strip().split()\n        if not toks or toks[0] not in ALLOWED:\n            bad = toks[0] if toks else \"\"\n            return (f\"ERROR: command not allowed: {bad}. Use grep, \"\n                    \"find, ls, cat, head, tail, sed -n, wc.\")\n    try:\n        p = subprocess.run(cmd, shell=True, cwd=f\"{SRCROOT}/{iid}\",\n                           capture_output=True, text=True, timeout=20)\n        out = p.stdout + p.stderr\n    except subprocess.TimeoutExpired:\n        return \"ERROR: command timed out after 20s.\"\n    return clip(out) or \"(no output)\"\n\ndef read_file(iid, path, start_line=None, end_line=None):\n    root = os.path.realpath(f\"{SRCROOT}/{iid}\")\n    rel = str(path).strip().lstrip(\"/\")\n    for pre in (\"workspace/\", \"testbed/\", \"repo/\"):\n        if rel.startswith(pre):\n            rel = rel[len(pre):]\n    p = os.path.realpath(os.path.join(root, rel))\n    if not p.startswith(root + os.sep) or not os.path.isfile(p):\n        return f\"ERROR: file not found: {path}\"\n    with open(p, errors=\"replace\") as f:\n        lines = f.read().splitlines()\n    s = max(1, int(start_line or 1))\n    e = int(end_line or s + 149)\n    e = min(e, s + 199, len(lines))\n    body = \"\\n\".join(f\"{i}: {lines[i - 1]}\" for i in range(s, e + 1))\n    return clip(f\"{rel} lines {s}-{e} of {len(lines)}\\n{body}\", 6000)\n\nprint(\"PART 1 OK | tasks:\", len(TASKS), \"| labels:\", len(LABELS))\n\n\n# ===== PROBE PART 2: graph tools (same code path for every world) =====\nfrom collections import OrderedDict\nfrom swegemma import graph as sg\n\n_gcache, _glock = OrderedDict(), threading.Lock()\n\ndef world_graph(world, iid):\n    key = (world, iid)\n    with _glock:\n        if key in _gcache:\n            _gcache.move_to_end(key)\n            return _gcache[key]\n    t = TASKS[iid]\n    gd, ed = WORLDS[world]\n    g = sg.get_graph(repo_name=t[\"repo\"], graph_dir=gd,\n                     embeddings_dir=ed, base_commit=t[\"base_commit\"])\n    ids = {strip(n): n for n in g.nodes}\n    with _glock:\n        _gcache[key] = (g, ids)\n        while len(_gcache) > 48:\n            _gcache.popitem(last=False)\n    return g, ids\n\ndef resolve_node(ids, q):\n    q = strip(q).strip(\"`() \")\n    if q in ids:\n        return q, None\n    hits = [s for s in ids if s.endswith(\".\" + q)]\n    if len(hits) == 1:\n        return hits[0], None\n    if hits:\n        return None, (\"Ambiguous name. Candidates: \"\n                      + \", \".join(sorted(hits)[:10]))\n    return None, (f\"Symbol not found in graph: {q}. Use \"\n                  \"search_similar_code or grep to find the dotted path.\")\n\ndef tool_neighbors(world, iid, node):\n    g, ids = world_graph(world, iid)\n    nid, err = resolve_node(ids, node)\n    if err:\n        return err\n    nb = sg.get_neighbor(node=nid, graph=g, max_neighbors=20)\n    names = sorted({strip(x) for x in nb} - {nid})\n    return json.dumps({\"node\": nid, \"neighbors\": names})\n\ndef tool_similar(world, iid, query, k=10):\n    t = TASKS[iid]\n    gd, ed = WORLDS[world]\n    g, ids = world_graph(world, iid)\n    try:\n        k = max(1, min(int(k or 10), 20))\n    except Exception:\n        k = 10\n    res = sg.get_similar_nodes(node=strip(query), repo_name=t[\"repo\"],\n                               k=k, graph=g, graph_dir=gd,\n                               embeddings_dir=ed,\n                               base_commit=t[\"base_commit\"])\n    out = []\n    for r in res:\n        if isinstance(r, dict):\n            sim = r.get(\"similarity\", r.get(\"score\"))\n            if isinstance(sim, (int, float)):\n                sim = round(float(sim), 4)\n            out.append({\"node\": strip(r.get(\"node_name\")),\n                        \"similarity\": sim})\n        else:\n            out.append({\"node\": strip(r)})\n    return json.dumps(out)\n\ndef tool_subgraph(world, iid, nodes):\n    g, ids = world_graph(world, iid)\n    if isinstance(nodes, str):\n        nodes = [x for x in re.split(r\"[,\\s]+\", nodes) if x]\n    good, errs = [], []\n    for q in list(nodes)[:15]:\n        nid, err = resolve_node(ids, q)\n        if nid:\n            good.append(nid)\n        else:\n            errs.append(err)\n    sub = g.subgraph([ids[s] for s in good])\n    edges = sorted({tuple(sorted((strip(u), strip(v))))\n                    for u, v in sub.edges() if u != v})\n    return json.dumps({\"nodes\": good, \"edges\": edges, \"errors\": errs})\n\ndef clear_graphs():\n    with _glock:\n        _gcache.clear()\n\nprint(\"PART 2 OK\")\n\n\n# ===== PROBE PART 2b: one graph call at a time (thread-safety fix) =====\nSG_LOCK = threading.RLock()\nif not hasattr(sg, \"_orig\"):\n    sg._orig = (sg.get_graph, sg.get_neighbor, sg.get_similar_nodes)\n\ndef _locked(f):\n    def g(*a, **k):\n        with SG_LOCK:\n            return f(*a, **k)\n    return g\n\nsg.get_graph = _locked(sg._orig[0])\nsg.get_neighbor = _locked(sg._orig[1])\nsg.get_similar_nodes = _locked(sg._orig[2])\n\ndef world_graph(world, iid):\n    key = (world, iid)\n    with SG_LOCK:\n        if key in _gcache:\n            _gcache.move_to_end(key)\n            return _gcache[key]\n        t = TASKS[iid]\n        gd, ed = WORLDS[world]\n        g = sg.get_graph(repo_name=t[\"repo\"], graph_dir=gd,\n                         embeddings_dir=ed,\n                         base_commit=t[\"base_commit\"])\n        ids = {strip(n): n for n in g.nodes}\n        _gcache[key] = (g, ids)\n        while len(_gcache) > 48:\n            _gcache.popitem(last=False)\n        return g, ids\n\nprint(\"PART 2b OK | graph calls are serialized\")\n\n\n# ===== PROBE PART 3: tool schemas and prompts =====\ndef fn(name, desc, props, req):\n    return {\"type\": \"function\", \"function\": {\n        \"name\": name, \"description\": desc,\n        \"parameters\": {\"type\": \"object\", \"properties\": props,\n                       \"required\": req}}}\n\nS = {\"type\": \"string\"}\nI = {\"type\": \"integer\"}\nL = {\"type\": \"array\", \"items\": {\"type\": \"string\"}}\nT_RUN = fn(\"run_command\",\n           \"Run a read-only shell command (grep, find, ls, cat, head, \"\n           \"tail, sed -n, wc) in the repository root.\",\n           {\"command\": S}, [\"command\"])\nT_READ = fn(\"read_file\",\n            \"Read lines of a file. Paths are relative to the \"\n            \"repository root.\",\n            {\"path\": S, \"start_line\": I, \"end_line\": I}, [\"path\"])\nT_SIM = fn(\"search_similar_code\",\n           \"Find functions/classes with similar precomputed embeddings \"\n           \"given a symbol name (e.g. pkg.module.Class.method, \"\n           \"Class.method, or func_name).\",\n           {\"query\": S, \"k\": I}, [\"query\"])\nT_NB = fn(\"get_code_neighbors\",\n          \"List the callers and callees of a symbol in the pre-built \"\n          \"code graph.\", {\"node\": S}, [\"node\"])\nT_SUB = fn(\"get_code_subgraph\",\n           \"Return the call edges among a list of symbols in the \"\n           \"pre-built code graph.\", {\"nodes\": L}, [\"nodes\"])\nT_SUBMIT = fn(\"submit_localization\",\n              \"Submit the final answer: up to 5 fully qualified \"\n              \"function or method names (dotted paths such as \"\n              \"pkg.module.Class.method) whose code must change to \"\n              \"resolve the issue, most likely first.\",\n              {\"functions\": L}, [\"functions\"])\n\nBASE_SYS = (\n    \"You are an expert software engineer localizing a GitHub issue. \"\n    \"The repository {repo} is checked out at the commit where the \"\n    \"issue was reported; the checkout contains its Python source \"\n    \"files. Find the functions or methods whose code must change to \"\n    \"resolve the issue. Investigate with the tools, one tool call at \"\n    \"a time. When you are confident, call submit_localization with \"\n    \"up to 5 fully qualified dotted names (for example \"\n    \"package.module.Class.method), most likely first. You have at \"\n    \"most {budget} tool calls.\")\nGRAPH_NOTE = (\n    \"\\n\\nThis repository has pre-built code graph and embedding data \"\n    \"indexed by dotted symbol path (e.g. pkg.module.Class.method). \"\n    \"Use these tools for fast, targeted navigation: \"\n    \"search_similar_code, get_code_neighbors, get_code_subgraph.\")\nNR_SYS = (\n    \"You are an expert software engineer. You cannot see the code. \"\n    \"Using only the issue text and what you know about the \"\n    \"repository {repo}, guess which functions or methods must change \"\n    \"to resolve the issue. Call submit_localization with up to 5 \"\n    \"fully qualified dotted names, most likely first.\")\nNUDGE = (\"Please continue by calling one of the tools. When you are \"\n         \"confident, call submit_localization.\")\nFORCE = (\"Your tool budget is used up. Call submit_localization now \"\n         \"with your best answer.\")\nGRAPH_TOOLS = {\"search_similar_code\", \"get_code_neighbors\",\n               \"get_code_subgraph\"}\n\ndef tools_for(world):\n    if world == \"NR\":\n        return [T_SUBMIT]\n    t = [T_RUN, T_READ]\n    if world in WORLDS:\n        t += [T_SIM, T_NB, T_SUB]\n    return t + [T_SUBMIT]\n\nprint(\"PART 3 OK\")\n\n\n# ===== PROBE PART 4: model client and episode loop =====\nimport requests\n\nROOT = server_instance.base_url.rstrip(\"/\")\nROOT = ROOT[:-3] if ROOT.endswith(\"/v1\") else ROOT\nH = {\"Authorization\": \"Bearer EMPTY\"}\nMID = requests.get(ROOT + \"/v1/models\", headers=H,\n                   timeout=60).json()[\"data\"][0][\"id\"]\nMODEL_TAG = MID.rstrip(\"/\").split(\"/\")[-2]\n\ndef chat(messages, tools, seed, temperature=0.7, max_tokens=2048):\n    p = {\"model\": MID, \"messages\": messages, \"tools\": tools,\n         \"tool_choice\": \"auto\", \"temperature\": temperature,\n         \"max_tokens\": max_tokens, \"seed\": seed}\n    r = requests.post(ROOT + \"/v1/chat/completions\", json=p,\n                      headers=H, timeout=600).json()\n    if \"choices\" not in r:\n        raise RuntimeError(str(r)[:300])\n    return r\n\ndef dispatch(world, iid, name, a):\n    if name == \"run_command\":\n        return run_command(iid, a.get(\"command\", \"\"))\n    if name == \"read_file\":\n        return read_file(iid, a.get(\"path\", \"\"),\n                         a.get(\"start_line\"), a.get(\"end_line\"))\n    if world in WORLDS and name == \"search_similar_code\":\n        return tool_similar(world, iid, a.get(\"query\", \"\"),\n                            a.get(\"k\", 10))\n    if world in WORLDS and name == \"get_code_neighbors\":\n        return tool_neighbors(world, iid, a.get(\"node\", \"\"))\n    if world in WORLDS and name == \"get_code_subgraph\":\n        return tool_subgraph(world, iid, a.get(\"nodes\", []))\n    return f\"ERROR: unknown tool {name}\"\n\ndef parse_pred(a):\n    fs = a.get(\"functions\", []) if isinstance(a, dict) else []\n    if isinstance(fs, str):\n        fs = re.split(r\"[,\\s]+\", fs)\n    return [strip(x) for x in fs if str(x).strip()][:5]\n\ndef run_episode(iid, world, seed, budget=20):\n    t = TASKS[iid]\n    sysp = (NR_SYS if world == \"NR\" else BASE_SYS).format(\n        repo=t[\"repo\"], budget=budget)\n    if world in WORLDS:\n        sysp += GRAPH_NOTE\n    msgs = [{\"role\": \"system\", \"content\": sysp},\n            {\"role\": \"user\",\n             \"content\": \"Issue:\\n\" + t[\"problem_statement\"][:6000]}]\n    log = {\"id\": iid, \"world\": world, \"seed\": seed,\n           \"model\": MODEL_TAG, \"budget\": budget, \"steps\": [],\n           \"pred\": [], \"status\": \"\", \"nudges\": 0,\n           \"max_prompt_tokens\": 0, \"completion_tokens\": 0}\n    T0, calls, forced = time.time(), 0, False\n    tools = tools_for(world)\n    while True:\n        if (calls >= budget or log[\"max_prompt_tokens\"] > 26000) \\\n                and not forced:\n            forced = True\n            tools = [T_SUBMIT]\n            msgs.append({\"role\": \"user\", \"content\": FORCE})\n        try:\n            r = chat(msgs, tools, seed * 1000 + calls + log[\"nudges\"])\n        except Exception as e:\n            log[\"status\"], log[\"error\"] = \"API_ERROR\", str(e)[:300]\n            break\n        u = r.get(\"usage\", {})\n        log[\"max_prompt_tokens\"] = max(log[\"max_prompt_tokens\"],\n                                       u.get(\"prompt_tokens\", 0))\n        log[\"completion_tokens\"] += u.get(\"completion_tokens\", 0)\n        m = r[\"choices\"][0][\"message\"]\n        tcs = m.get(\"tool_calls\") or []\n        if not tcs:\n            if log[\"nudges\"] >= 2 or forced:\n                log[\"status\"] = \"NO_SUBMIT\"\n                break\n            log[\"nudges\"] += 1\n            msgs.append({\"role\": \"assistant\",\n                         \"content\": m.get(\"content\") or \"\"})\n            msgs.append({\"role\": \"user\", \"content\": NUDGE})\n            continue\n        tc = tcs[0]\n        name = tc[\"function\"][\"name\"]\n        try:\n            a = json.loads(tc[\"function\"][\"arguments\"] or \"{}\")\n        except Exception:\n            a = None\n        msgs.append({\"role\": \"assistant\",\n                     \"content\": m.get(\"content\") or \"\",\n                     \"tool_calls\": [tc]})\n        if a is not None and name == \"submit_localization\":\n            log[\"pred\"] = parse_pred(a)\n            log[\"status\"] = \"SUBMIT_FORCED\" if forced else \"SUBMIT\"\n            break\n        calls += 1\n        if a is None:\n            out = \"ERROR: tool arguments were not valid JSON.\"\n        elif forced:\n            out = \"ERROR: only submit_localization is allowed now.\"\n        else:\n            try:\n                out = dispatch(world, iid, name, a)\n            except Exception as e:\n                out = f\"ERROR: {type(e).__name__}: {str(e)[:200]}\"\n        log[\"steps\"].append({\"tool\": name, \"args\": a, \"out\": out,\n                             \"t\": round(time.time() - T0, 1)})\n        msgs.append({\"role\": \"tool\", \"name\": name,\n                     \"tool_call_id\": tc.get(\"id\", f\"call{calls}\"),\n                     \"content\": out})\n        if forced:\n            log[\"forced_tries\"] = log.get(\"forced_tries\", 0) + 1\n            if log[\"forced_tries\"] >= 2:\n                log[\"status\"] = \"NO_SUBMIT\"\n                break\n    log[\"calls\"] = calls\n    log[\"seconds\"] = round(time.time() - T0, 1)\n    return log\n\nprint(\"PART 4 OK | model:\", MODEL_TAG)\n\n\n# ===== PROBE PART 5: scoring, batch runner, summary =====\nimport concurrent.futures as cf\n\n_files = {}\ndef py_files(iid):\n    if iid not in _files:\n        root = f\"{SRCROOT}/{iid}\"\n        out = []\n        for d, _, fs in os.walk(root):\n            out += [os.path.relpath(f\"{d}/{x}\", root)\n                    for x in fs if x.endswith(\".py\")]\n        _files[iid] = out\n    return _files[iid]\n\ndef resolve_file(iid, nid):\n    parts = nid.split(\".\")\n    for i in range(len(parts), 0, -1):\n        mod = \"/\".join(parts[:i])\n        hits = [f for f in py_files(iid)\n                if f == mod + \".py\" or f.endswith(\"/\" + mod + \".py\")\n                or f == mod + \"/__init__.py\"\n                or f.endswith(\"/\" + mod + \"/__init__.py\")]\n        if hits:\n            return min(hits, key=len)\n    return None\n\ndef norm(p):\n    p = strip(p).strip(\"`'\\\"() \").replace(\"::\", \".\")\n    p = p.replace(\".py.\", \".\").replace(\"/\", \".\")\n    return p[:-3] if p.endswith(\".py\") else p\n\ndef match(pred, gold):\n    p = norm(pred)\n    if not p:\n        return False\n    if p == gold or p.endswith(\".\" + gold):\n        return True\n    return \".\" in p and gold.endswith(\".\" + p)\n\ndef score(log):\n    lab = LABELS[log[\"id\"]]\n    gold, preds = lab[\"gold\"], log[\"pred\"]\n    log[\"hit5\"] = any(match(p, g) for p in preds for g in gold)\n    log[\"hit1\"] = bool(preds) and any(match(preds[0], g) for g in gold)\n    gf = {resolve_file(log[\"id\"], g) for g in gold} - {None}\n    pf = {resolve_file(log[\"id\"], norm(p)) for p in preds} - {None}\n    log[\"file_hit5\"] = bool(gf & pf)\n    st = log[\"steps\"]\n    log[\"graph_calls\"] = sum(s[\"tool\"] in GRAPH_TOOLS for s in st)\n    seen = [i for i, s in enumerate(st) if s[\"tool\"] in GRAPH_TOOLS\n            and any(g in str(s[\"out\"]) for g in gold)]\n    log[\"graph_exposed\"] = bool(seen)\n    log[\"exposed_then_hit\"] = bool(seen) and log[\"hit5\"]\n    log[\"primary\"] = lab[\"primary\"]\n    log[\"hop\"] = lab[\"hop\"]\n    log[\"n_anchors\"] = lab[\"n_anchors\"]\n    return log\n\ndef safe_episode(iid, world, seed, budget):\n    try:\n        return run_episode(iid, world, seed, budget)\n    except Exception as e:\n        return {\"id\": iid, \"world\": world, \"seed\": seed,\n                \"model\": MODEL_TAG, \"budget\": budget, \"steps\": [],\n                \"pred\": [], \"status\": \"CRASH\", \"calls\": 0,\n                \"seconds\": 0, \"error\": f\"{type(e).__name__}: {e}\"[:300]}\n\ndef run_batch(ids, worlds, seeds, out_path, budget=20, workers=24):\n    done = set()\n    if os.path.exists(out_path):\n        with open(out_path) as f:\n            for line in f:\n                d = json.loads(line)\n                if d[\"status\"] not in (\"CRASH\", \"API_ERROR\"):\n                    done.add((d[\"id\"], d[\"world\"], d[\"seed\"]))\n    jobs = [(i, w, s) for i in ids for w in worlds for s in seeds\n            if (i, w, s) not in done]\n    print(\"jobs to run:\", len(jobs), \"| already done:\", len(done))\n    T0, n = time.time(), 0\n    with cf.ThreadPoolExecutor(workers) as ex, \\\n            open(out_path, \"a\") as f:\n        futs = [ex.submit(safe_episode, *j, budget) for j in jobs]\n        for fu in cf.as_completed(futs):\n            r = fu.result()\n            log = r if r[\"status\"] == \"CRASH\" else score(r)\n            f.write(json.dumps(log) + \"\\n\")\n            f.flush()\n            n += 1\n            if n % 20 == 0 or n == len(jobs):\n                print(f\"  {n}/{len(jobs)} | {round(time.time() - T0)} s\")\n    clear_graphs()\n\ndef load_runs(path):\n    with open(path) as f:\n        rows = [json.loads(x) for x in f]\n    last = {}\n    for r in rows:\n        last[(r[\"id\"], r[\"world\"], r[\"seed\"])] = r\n    return list(last.values())\n\ndef summary(path):\n    rs = load_runs(path)\n    by = collections.defaultdict(list)\n    for r in rs:\n        by[r[\"world\"]].append(r)\n    print(f\"{'world':6}{'n':>5}{'hit@5':>7}{'hit@1':>7}{'file@5':>8}\"\n          f\"{'calls':>7}{'graph%':>8}{'exposed':>9}{'sec':>6}\")\n    for w in sorted(by):\n        x = by[w]\n        n = len(x)\n        mean = lambda k: sum(float(r.get(k) or 0) for r in x) / n\n        calls = sum(r.get(\"calls\", 0) for r in x)\n        gshare = sum(r.get(\"graph_calls\", 0) for r in x) / max(calls, 1)\n        print(f\"{w:6}{n:>5}{mean('hit5'):>7.2f}{mean('hit1'):>7.2f}\"\n              f\"{mean('file_hit5'):>8.2f}{mean('calls'):>7.1f}\"\n              f\"{gshare:>8.2f}{mean('graph_exposed'):>9.2f}\"\n              f\"{mean('seconds'):>6.0f}\")\n    st = collections.Counter((r[\"world\"], r[\"status\"]) for r in rs)\n    print(\"status:\", sorted(st.items()))\n    errs = [r.get(\"error\") for r in rs if r.get(\"error\")]\n    if errs:\n        print(\"example error:\", errs[0])\n\nprint(\"PART 5 OK\")\n\n\n# ===== PROBE PART 7: graph-first arm (patch only, no run) =====\nimport inspect\nGRAPH_FIRST = (\n    \"\\n\\nRequired workflow: first call search_similar_code on symbol \"\n    \"names from the issue, then call get_code_neighbors on the most \"\n    \"relevant results to follow callers and callees, and only then \"\n    \"read files to confirm. Base your answer on what the graph shows.\")\nfor w in (\"W1\", \"W2\", \"W3\"):\n    WORLDS[w + \"G\"] = WORLDS[w]\n_src = inspect.getsource(run_episode)\n_old = \"        sysp += GRAPH_NOTE\\n\"\n_new = (\"        sysp += GRAPH_NOTE + (GRAPH_FIRST if world.endswith('G')\"\n        \" else '')\\n\")\nassert _old in _src, \"patch point not found\"\nexec(_src.replace(_old, _new))\nassert \"GRAPH_FIRST\" in run_episode.__code__.co_names\nprint(\"PART 7 OK | worlds:\", sorted(WORLDS))\n\n\n# ===== PROBE PART S: E4B extra seeds for the real vs rewired contrasts =====\nassert \"e4b\" in MODEL_TAG, \"Wrong model: \" + MODEL_TAG\nPRIMARY = sorted(r[\"id\"] for r in LABELS.values() if r[\"primary\"])\nassert len(PRIMARY) == 82, len(PRIMARY)\nORDER = list(PRIMARY)\nrandom.Random(20261113).shuffle(ORDER)\nS_WORLDS = [\"W1\", \"W2\", \"W1G\", \"W2G\"]\nS_SEEDS = [3, 4, 5, 6, 7, 8]\nOUT_S = f\"/kaggle/working/extra_{MODEL_TAG}.jsonl\"\nLIMIT_H, CHUNK, WORKERS, MAX_RESTARTS = 7.0, 8, 24, 8\nT_BEGIN = time.time()\nrestarts = 0\n\ndef alive():\n    try:\n        r = requests.get(ROOT + \"/v1/models\", headers=H, timeout=20)\n        return r.status_code == 200\n    except Exception:\n        return False\n\ndef restart_server():\n    global server_instance\n    try:\n        server_instance.stop()\n    except Exception as e:\n        print(\"stop failed:\", type(e).__name__)\n    time.sleep(30)\n    server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)\n    server_instance.start()\n    print(\"server restarted | alive:\", alive())\n\ndef n_ok(chunk, seed):\n    if not os.path.exists(OUT_S):\n        return 0\n    return sum(1 for r in load_runs(OUT_S)\n               if r[\"id\"] in chunk and r[\"seed\"] == seed\n               and r[\"status\"] not in (\"API_ERROR\", \"CRASH\"))\n\nprint(\"tasks:\", len(ORDER), \"| episodes planned:\",\n      len(ORDER) * len(S_WORLDS) * len(S_SEEDS))\nseeds_done, stop = [], False\nfor seed in S_SEEDS:\n    ok_seed = 0\n    for k in range(0, len(ORDER), CHUNK):\n        hours = (time.time() - T_BEGIN) / 3600\n        if hours > LIMIT_H:\n            print(\"TIME GUARD: stopping at\", round(hours, 2), \"h\")\n            stop = True\n            break\n        if restarts > MAX_RESTARTS:\n            print(\"TOO MANY RESTARTS: stopping\")\n            stop = True\n            break\n        chunk = ORDER[k:k + CHUNK]\n        want = len(chunk) * len(S_WORLDS)\n        for attempt in range(3):\n            if not alive():\n                restarts += 1\n                print(\"server dead, restart\", restarts)\n                try:\n                    restart_server()\n                except Exception as e:\n                    print(\"restart failed:\", type(e).__name__, str(e)[:200])\n                    break\n            run_batch(chunk, S_WORLDS, [seed], OUT_S,\n                      budget=20, workers=WORKERS)\n            if n_ok(chunk, seed) == want:\n                break\n        ok_seed += n_ok(chunk, seed)\n    if stop:\n        break\n    full = ok_seed == len(ORDER) * len(S_WORLDS)\n    if full:\n        seeds_done.append(seed)\n    print(\"== seed\", seed, \"| ok episodes:\", ok_seed, \"/\",\n          len(ORDER) * len(S_WORLDS), \"| complete:\", full,\n          \"| hours:\", round((time.time() - T_BEGIN) / 3600, 2),\n          \"| restarts:\", restarts)\nsummary(OUT_S)\ntry:\n    server_instance.stop()\nexcept Exception:\n    pass\nprint(\"PART S DONE | complete seeds:\", seeds_done, \"| restarts:\", restarts)\n\n\n# ===== PROBE PART T: secondary task set (labeled, non-primary tasks) =====\nSECONDARY = sorted(r[\"id\"] for r in LABELS.values()\n                   if r[\"gold\"] and not r[\"primary\"])\nprint(\"model:\", MODEL_TAG, \"| secondary tasks:\", len(SECONDARY))\nORDER_T = list(SECONDARY)\nrandom.Random(20261114).shuffle(ORDER_T)\nT_WORLDS = [\"W0\", \"W1\", \"W2\", \"W1G\", \"W2G\"]\nT_SEEDS = [1, 2]\nPER_TASK = len(T_WORLDS) * len(T_SEEDS)\nOUT_T = f\"/kaggle/working/secondary_{MODEL_TAG}.jsonl\"\nWORKERS_T = 16 if \"31b\" in MODEL_TAG else 24\nLIMIT_H, CHUNK, MAX_RESTARTS = 4.0, 4, 8\nT_BEGIN = time.time()\nrestarts = 0\n\ndef alive():\n    try:\n        r = requests.get(ROOT + \"/v1/models\", headers=H, timeout=20)\n        return r.status_code == 200\n    except Exception:\n        return False\n\ndef restart_server():\n    global server_instance\n    try:\n        server_instance.stop()\n    except Exception as e:\n        print(\"stop failed:\", type(e).__name__)\n    time.sleep(30)\n    server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)\n    server_instance.start()\n    print(\"server restarted | alive:\", alive())\n\ndef n_ok(chunk):\n    if not os.path.exists(OUT_T):\n        return 0\n    return sum(1 for r in load_runs(OUT_T) if r[\"id\"] in chunk\n               and r[\"status\"] not in (\"API_ERROR\", \"CRASH\"))\n\nprint(\"episodes planned:\", len(ORDER_T) * PER_TASK,\n      \"| workers:\", WORKERS_T)\nfinished = 0\nfor k in range(0, len(ORDER_T), CHUNK):\n    hours = (time.time() - T_BEGIN) / 3600\n    if hours > LIMIT_H:\n        print(\"TIME GUARD: stopping at\", round(hours, 2), \"h\")\n        break\n    if restarts > MAX_RESTARTS:\n        print(\"TOO MANY RESTARTS: stopping\")\n        break\n    chunk = ORDER_T[k:k + CHUNK]\n    for attempt in range(3):\n        if not alive():\n            restarts += 1\n            print(\"server dead, restart\", restarts)\n            try:\n                restart_server()\n            except Exception as e:\n                print(\"restart failed:\", type(e).__name__, str(e)[:200])\n                break\n        run_batch(chunk, T_WORLDS, T_SEEDS, OUT_T,\n                  budget=20, workers=WORKERS_T)\n        if n_ok(chunk) == len(chunk) * PER_TASK:\n            break\n    finished += len(chunk)\n    print(\"== chunk done | ok episodes:\", n_ok(chunk), \"/\",\n          len(chunk) * PER_TASK, \"| tasks:\", finished, \"/\", len(ORDER_T),\n          \"| hours:\", round((time.time() - T_BEGIN) / 3600, 2),\n          \"| restarts:\", restarts)\nsummary(OUT_T)\ntry:\n    server_instance.stop()\nexcept Exception:\n    pass\nprint(\"PART T DONE | tasks attempted:\", finished, \"| restarts:\", restarts)\n\n\n# ===== PROBE PART H: held-out repositories (click, flask, attrs) =====\nHELD = sorted(LABELS)\nassert all(i.split(\"_\")[0] in (\"click\", \"flask\", \"attrs\") for i in HELD)\nprint(\"model:\", MODEL_TAG, \"| held-out tasks:\", len(HELD))\nORDER_T = list(HELD)\nrandom.Random(20261115).shuffle(ORDER_T)\nT_WORLDS = [\"W0\", \"W1\", \"W2\", \"W1G\", \"W2G\"]\nT_SEEDS = [1, 2]\nPER_TASK = len(T_WORLDS) * len(T_SEEDS)\nOUT_T = f\"/kaggle/working/heldout_{MODEL_TAG}.jsonl\"\nWORKERS_T = 16 if \"31b\" in MODEL_TAG else 24\nLIMIT_H, CHUNK, MAX_RESTARTS = 6.0, 4, 8\nT_BEGIN = time.time()\nrestarts = 0\n\ndef alive():\n    try:\n        r = requests.get(ROOT + \"/v1/models\", headers=H, timeout=20)\n        return r.status_code == 200\n    except Exception:\n        return False\n\ndef restart_server():\n    global server_instance\n    try:\n        server_instance.stop()\n    except Exception as e:\n        print(\"stop failed:\", type(e).__name__)\n    time.sleep(30)\n    server_instance = VllmServer(vllm_cfg, adapter_manifest=adapters)\n    server_instance.start()\n    print(\"server restarted | alive:\", alive())\n\ndef n_ok(chunk):\n    if not os.path.exists(OUT_T):\n        return 0\n    return sum(1 for r in load_runs(OUT_T) if r[\"id\"] in chunk\n               and r[\"status\"] not in (\"API_ERROR\", \"CRASH\"))\n\nprint(\"episodes planned:\", len(ORDER_T) * PER_TASK,\n      \"| workers:\", WORKERS_T)\nfinished = 0\nfor k in range(0, len(ORDER_T), CHUNK):\n    hours = (time.time() - T_BEGIN) / 3600\n    if hours > LIMIT_H:\n        print(\"TIME GUARD: stopping at\", round(hours, 2), \"h\")\n        break\n    if restarts > MAX_RESTARTS:\n        print(\"TOO MANY RESTARTS: stopping\")\n        break\n    chunk = ORDER_T[k:k + CHUNK]\n    for attempt in range(3):\n        if not alive():\n            restarts += 1\n            print(\"server dead, restart\", restarts)\n            try:\n                restart_server()\n            except Exception as e:\n                print(\"restart failed:\", type(e).__name__, str(e)[:200])\n                break\n        run_batch(chunk, T_WORLDS, T_SEEDS, OUT_T,\n                  budget=20, workers=WORKERS_T)\n        if n_ok(chunk) == len(chunk) * PER_TASK:\n            break\n    finished += len(chunk)\n    print(\"== chunk done | ok episodes:\", n_ok(chunk), \"/\",\n          len(chunk) * PER_TASK, \"| tasks:\", finished, \"/\", len(ORDER_T),\n          \"| hours:\", round((time.time() - T_BEGIN) / 3600, 2),\n          \"| restarts:\", restarts)\nsummary(OUT_T)\ntry:\n    server_instance.stop()\nexcept Exception:\n    pass\nprint(\"PART H DONE | tasks attempted:\", finished, \"| restarts:\", restarts)\n"
with open(OUT + '/probe_harness.py' if 'OUT' in globals() else 'probe_harness.py', 'w') as f:
    f.write(PROBE_SOURCE)
print('probe harness source written, lines:', PROBE_SOURCE.count(chr(10)) + 1)